In [ ]:
from pathlib import Path
import os

import geopandas as gpd
import xarray as xr
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping

def crop_and_save_tif(src_raster_filepath,site,output_filename):
        
        os.makedirs(root / 'data' / site, exist_ok=True)
        
        b = gpd.read_file(root / 'data' / 'nyc_boundary.gpkg')

        t = xr.open_dataarray(src_raster_filepath)

        b = b.to_crs(t.rio.crs) # transform boundary bc it's faster than reprojecting a giant raster

        geom = [mapping(geom) for geom in b.geometry]

        # Open raster and clip to site boundary
        with rasterio.open(src_raster_filepath) as src:
            out_image, out_transform = mask(src, geom, crop=True)
            out_meta = src.meta.copy()

        # Update metadata
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        # Save to disk 
        with rasterio.open(root /  f"{output_filename}.tif", "w", **out_meta) as dest:
            dest.write(out_image)
        print(f'{site} file saved to: {root}/{output_filename}.tif')



root = Path.cwd()

site = 'nlcd'
src_raster_filepath = Path(r"C:\Users\roseh\Downloads\nlcd_tcc_CONUS_2021_v2023-5_wgs84\nlcd_tcc_conus_wgs84_v2023-5_20210101_20211231.tif")
output_filename = 'nlcd_tc_2021'


crop_and_save_tif(src_raster_filepath,site,output_filename)
